In [1]:
import os 
os.chdir(os.path.dirname(os.getcwd()))

In [2]:
import ast
import pandas as pd
from sklearn.metrics import f1_score as f1 
from sklearn.metrics import precision_recall_fscore_support as prf
import numpy as np

## Obitools

In [3]:


def can_family(tax_ids,database):

    sub = database[database['taxid_ncbi'].isin(tax_ids)]
    
    family = sub['family'].unique()

    if len(family) > 1:
        return 'NC_'
    return family[0]

acc = []
f1_scrores = []
p_score =[]
r_score =[]

marker = "berry"

OBI_pred_path = f'Obitools/scripts/results/obi3/{marker}'



for fold in range(1,7):

    print("-----------------")
    database = pd.read_csv(f'Obitools/scripts/data/{marker}/folds/fold_{fold}/train.csv')

    preds = pd.read_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3.csv', sep="\t")



    preds['BEST_MATCH_IDS'] = preds['BEST_MATCH_IDS'].apply(ast.literal_eval)  # Convert to list of strings
    preds['BEST_MATCH_TAXIDS'] = preds['BEST_MATCH_TAXIDS'].apply(ast.literal_eval) 

    preds['preds_family'] = preds['BEST_MATCH_TAXIDS'].apply(lambda x: can_family(x,database))

    labels = pd.read_csv(f'Obitools/scripts/data/{marker}/folds/fold_{fold}/test.csv')

    classes = labels['family'].unique()

    preds['family_gt'] = labels['family']


    p,r,f,s = prf(preds["family_gt"], preds["preds_family"], average="macro", labels=classes, zero_division=0)
    print(f'Fold {fold} f1: {f}')
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')

    preds = preds[['family_gt', 'preds_family', 'BEST_IDENTITY','BEST_MATCH_IDS', 'NUC_SEQ']]
    preds.to_csv(OBI_pred_path +f'/fold_{fold}_test_ecotag3_pred.csv', sep="\t", index=False)
    





-----------------
Fold 1 f1: 0.6779223955115896
Fold 1 precision: 0.6821704433895438
Fold 1 recall: 0.7368935391037696
-----------------
Fold 2 f1: 0.7033169947706602
Fold 2 precision: 0.6956943871247862
Fold 2 recall: 0.760468981759425
-----------------
Fold 3 f1: 0.6879276763551613
Fold 3 precision: 0.6945735280306891
Fold 3 recall: 0.743325150345117
-----------------
Fold 4 f1: 0.6944733879594925
Fold 4 precision: 0.6983904752264256
Fold 4 recall: 0.7533284061048534
-----------------
Fold 5 f1: 0.6860849302747718
Fold 5 precision: 0.6871381520993105
Fold 5 recall: 0.7372282166192903
-----------------
Fold 6 f1: 0.6908055370452941
Fold 6 precision: 0.6902776467813602
Fold 6 recall: 0.7423578351371872


## DNABERT_2

In [5]:
padnas_macro= []
F = []
for fold in [1,2,3,4,5,6]:
    print("-----------------")
    preds = pd.read_csv(f'DNABert2/experiments/fine_tune_taxa/outputs/teleo_from_pt_dnabert2/checkpoints/fold_{fold}/predictions.csv')
    
    print("fold",fold)
    p,r,f,s = prf(preds['labels_family'],preds['preds_family'],average='macro', labels=preds['labels_family'].unique(), zero_division=0)
    print('p',p,'r',r,'f',f)

    

-----------------
fold 1
p 0.4624058084231881 r 0.5450555628339964 f 0.4753844532609013
-----------------
fold 2
p 0.4250476387342441 r 0.5388035197970305 f 0.44833477605387456
-----------------
fold 3
p 0.44978696201619983 r 0.5337562328908297 f 0.4589478709956673
-----------------
fold 4
p 0.4567443750563477 r 0.5417447409456781 f 0.4715022455404731
-----------------
fold 5
p 0.4363354579849877 r 0.5381961833741391 f 0.4529424473615948
-----------------
fold 6
p 0.4629790582205139 r 0.5450930680914341 f 0.4774286683571579


## MMSeq2


In [13]:

for fold in [1,2,3,4,5,6]:

    preds_df = pd.read_csv(f'MMseqs2/result/teleov2_no_cefe/folds/fold_{fold}/tax_out/tax_out_lca.tsv', sep="\t", header=None)
    ground_truth = pd.read_csv(f'MMseqs2/teleov2_no_cefe/folds/fold_{fold}/test.csv')
    print("====================")
    print("fold",fold)

    preds_df[0] = preds_df[0].apply(lambda x: x.split('_')[-1]) # remove the prefix

    p,r,f,s = prf(preds_df[3],preds_df[0],average='micro',labels=ground_truth['family'].unique(),zero_division=0)
    print('p',p,'r',r,'f',f)

    


fold 1
p 0.5507399577167019 r 0.6504369538077404 f 0.596451058958214
fold 2
p 0.5764006791171478 r 0.6729435084241824 f 0.6209419295839049
fold 3
p 0.6085011185682326 r 0.6974358974358974 f 0.6499402628434886
fold 4
p 0.6358325219084713 r 0.7279821627647715 f 0.6787941787941788
fold 5
p 0.5704286964129484 r 0.6559356136820925 f 0.6102012166588676
fold 6
p 0.5528364849833148 r 0.637997432605905 f 0.5923718712753278


# Bertax

In [4]:

for fold in range(1,7):


    preds_pkl = f'/home/auguste/Desktop/bertax_training_2/teleo/fold_{fold}/test_multi_predictions.pkl'
    preds_pkl = pd.read_pickle(preds_pkl)

    print("====================")
    print("fold",fold)

    X_ground_truth, X_preds = preds_pkl['data'][0][1], preds_pkl['data'][1][1]
    labels= np.argmax(X_ground_truth, axis=1)
    preds = np.argmax(X_preds, axis=1)

    p,r,f,s = prf(labels,preds,average='macro',labels=np.unique(labels),zero_division=0)
    print('p',p,'r',r,'f',f)

    

fold 1
p 0.3280467516311281 r 0.39546326839603685 f 0.3305565978775961
fold 2
p 0.30106974771782874 r 0.40606365887871043 f 0.31875497324012436
fold 3
p 0.3190220342900245 r 0.39956105919717205 f 0.32214780190659986
fold 4
p 0.3419323581893524 r 0.4063336889909794 f 0.33760352982551317
fold 5
p 0.26718714923745823 r 0.3543141614977354 f 0.27656255666067103
fold 6
p 0.317377158522447 r 0.38370846201728553 f 0.32078768701696875


# Obitools 4.4.0

In [19]:
import json
import csv


for fold in range(1,7):
    # Ouvre le fichier d'entrée
    with open(f"/home/auguste/Desktop/eDNA/TeleoClassification/scripts/Obitools/scripts/results/obi4/berry/fold_{fold}/test.ecotag", "r") as infile:
        lines = infile.readlines()

    # Variables temporaires
    records = []
    current_record = {}

    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            # Sauvegarde l'ancien record si existant
            if current_record:
                records.append(current_record)
                current_record = {}

            # Sépare l'identifiant de la partie JSON
            header, metadata = line[1:].split(" ", 1)
            current_record["ID"] = header
            try:
                meta_dict = json.loads(metadata)
                for key, value in meta_dict.items():
                    current_record[key] = value
            except json.JSONDecodeError:
                print(f"Erreur de parsing JSON pour : {metadata}")
        elif line:
            # On assume que la séquence arrive en une seule ligne (ou on concatène sinon)
            current_record["sequence"] = current_record.get("sequence", "") + line

    # N'oublie pas d'ajouter le dernier record
    if current_record:
        records.append(current_record)

    # Détermine toutes les colonnes pour bien formater le CSV
    all_keys = set()
    for rec in records:
        all_keys.update(rec.keys())

    all_keys = sorted(all_keys)

    # Écriture du CSV
    with open(f"/home/auguste/Desktop/eDNA/TeleoClassification/scripts/Obitools/scripts/results/obi4/berry/fold_{fold}/output.csv", "w", newline="") as outfile:
        writer = csv.DictWriter(outfile, fieldnames=all_keys)
        writer.writeheader()
        for rec in records:
            writer.writerow(rec)

    print("Conversion terminée.")

Conversion terminée.
Conversion terminée.
Conversion terminée.
Conversion terminée.
Conversion terminée.
Conversion terminée.


In [34]:

def get_family(x,database):

    sub = database[database['genus']== x]

    family = sub['family'].unique()

    if len(family) > 1:
        return 'NC_'
    return family[0]

for fold in range(1,7):

    preds = pd.read_csv(f'/home/auguste/Desktop/eDNA/TeleoClassification/scripts/Obitools/scripts/results/obi4/Ac16/fold_{fold}/output.csv', sep=",")
    db = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/train.csv')
    labels = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/test.csv')

    preds['best_match']= preds['obitag_bestmatch'].apply(lambda x: '.'.join(x.split('.')[0:-1])) # remove the prefix

    preds['preds_family'] = preds['best_match'].apply(lambda x: get_family(x, db))

    preds['family_gt'] = labels['family']
    classes = labels['family'].unique()
    p, r, f, s = prf(preds["family_gt"], preds["preds_family"], average="macro", labels=classes, zero_division=0)
    print(f'Fold {fold} f1: {f}')
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')
    preds = preds[['family_gt', 'preds_family', 'obitag_bestid']]
    # preds.to_csv(f'fold_{fold}_output_pred.csv', sep=",", index=False)
    print("--------------------------------")

Fold 1 f1: 0.7335851110565166
Fold 1 precision: 0.7257589094798398
Fold 1 recall: 0.7888866759948661
--------------------------------
Fold 2 f1: 0.7344406252296806
Fold 2 precision: 0.727612457681178
Fold 2 recall: 0.7816897518932403
--------------------------------
Fold 3 f1: 0.7521754390900889
Fold 3 precision: 0.7380210078465893
Fold 3 recall: 0.8014494334261777
--------------------------------
Fold 4 f1: 0.7724507692060791
Fold 4 precision: 0.7621449881211785
Fold 4 recall: 0.8198835594707988
--------------------------------
Fold 5 f1: 0.7137156397223302
Fold 5 precision: 0.7035547824499825
Fold 5 recall: 0.7692901869591182
--------------------------------
Fold 6 f1: 0.7346532079351206
Fold 6 precision: 0.719948690951598
Fold 6 recall: 0.7860276190218051
--------------------------------


In [33]:

def get_family(x,database):

    sub = database[database['genus']== x]
    
    family = sub['family'].unique()
    
    if len(family) > 1:
        return 'NC_'
    return family[0]

def valid_rank(x):
    if x in ['family','subfamily', 'genus','subgenus','subspecies', 'species']:
        return True
    return False


for fold in range(1,7):

    preds = pd.read_csv(f'Obitools/scripts/results/obi4/Ac16/fold_{fold}/output.csv', sep=",")
    db = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/train.csv')
    labels = pd.read_csv(f'Obitools/scripts/data/Ac16/folds/fold_{fold}/test.csv')

    preds['best_match']= preds['obitag_bestmatch'].apply(lambda x: '.'.join(x.split('.')[0:-1])) # remove the prefix
    preds['preds_family'] = preds['best_match'].apply(lambda x: get_family(x, db))

    preds['obi_tag']= preds['taxid'].apply(lambda x: x.split('[')[1].split(']')[0]) # remove the prefix
    preds['valid_rank_preds'] = preds['obitag_rank'].apply(lambda x: valid_rank(x)) # remove the prefix

    preds['preds_family'] = preds.apply(lambda row: row['preds_family'] if row['valid_rank_preds'] else 'NC_', axis=1)

    preds['family_gt'] = labels['family']
    classes = labels['family'].unique()
    p, r, f, s = prf(preds["family_gt"], preds["preds_family"], average="macro", labels=classes, zero_division=0)
    
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')
    print(f'Fold {fold} f1: {f}')
    preds = preds[['family_gt', 'preds_family', 'obitag_bestid']]
    # preds.to_csv(f'fold_{fold}_output_pred.csv', sep=",", index=False)
    print("--------------------------------")

Fold 1 precision: 0.6223098929494278
Fold 1 recall: 0.564712298243643
Fold 1 f1: 0.575343972059691
--------------------------------
Fold 2 precision: 0.616677452533507
Fold 2 recall: 0.5846758088800394
Fold 2 f1: 0.5904770844681683
--------------------------------
Fold 3 precision: 0.6495643675876234
Fold 3 recall: 0.5935921029319379
Fold 3 f1: 0.6034681367450201
--------------------------------
Fold 4 precision: 0.6979063416010459
Fold 4 recall: 0.6644837965703113
Fold 4 f1: 0.6682308558418243
--------------------------------
Fold 5 precision: 0.5936923218899963
Fold 5 recall: 0.5624733019850329
Fold 5 f1: 0.5652758605737958
--------------------------------
Fold 6 precision: 0.6226706186350509
Fold 6 recall: 0.6003026748714788
Fold 6 f1: 0.6005848428132671
--------------------------------


# Ensemble : Obi4 classic + DnaBert


In [25]:

def get_family(x,database):

    sub = database[database['genus']== x]
    
    family = sub['family'].unique()

    if len(family) > 1:
        return 'NC_'
    return family[0]

def valid_rank(x):
    if x in ['family','subfamily', 'genus','subgenus','subspecies', 'species']:
        return True
    return False


for fold in range(1,7):

    preds_bert = pd.read_csv(f'DNABert2/experiments/fine_tune_taxa/outputs/teleo_from_mlm_no_cefe/checkpoints/fold_{fold}/predictions.csv')


    preds_obi = pd.read_csv(f'Obitools/scripts/results/obi4/teleo/fold_{fold}_output.csv', sep=",")
    db = pd.read_csv(f'Obitools/scripts/data/teleo/folds/fold_{fold}/train.csv')
    labels = pd.read_csv(f'Obitools/scripts/data/teleo/folds/fold_{fold}/test.csv')

    preds_obi['best_match']= preds_obi['obitag_bestmatch'].apply(lambda x: x.split('.')[0]) # remove the prefix
    preds_obi['preds_family'] = preds_obi['best_match'].apply(lambda x: get_family(x, db))

    preds_obi['obi_tag']= preds_obi['taxid'].apply(lambda x: x.split('[')[1].split(']')[0]) # remove the prefix
    preds_obi['valid_rank_preds'] = preds_obi['obitag_rank'].apply(lambda x: valid_rank(x)) # remove the prefix


    #choice preds_bert if not valid_rank_preds
    preds_obi['preds_family'] = preds_obi.apply(lambda row: row['preds_family'] if row['valid_rank_preds'] else preds_bert['preds_family_name'][row.name], axis=1)

    # preds['preds_family'] = preds.apply(lambda row: row['preds_family'] if row['valid_rank_preds'] else , axis=1)

    preds_obi['family_gt'] = labels['family']
    classes = labels['family'].unique()
    p, r, f, s = prf(preds_obi["family_gt"], preds_obi["preds_family"], average="macro", labels=classes, zero_division=0)
    print(f'Fold {fold} f1: {f}')
    print(f'Fold {fold} precision: {p}')
    print(f'Fold {fold} recall: {r}')
    preds = preds_obi[['family_gt', 'preds_family', 'obitag_bestid']]
    preds.to_csv(f'fold_{fold}_output_pred_ensemble_dnabert2.csv', sep=",", index=False)
    print("--------------------------------")

Fold 1 f1: 0.4956346703355422
Fold 1 precision: 0.4794261015293821
Fold 1 recall: 0.5662259640954032
--------------------------------
Fold 2 f1: 0.47202928857493337
Fold 2 precision: 0.4553985864928991
Fold 2 recall: 0.5575993137354633
--------------------------------
Fold 3 f1: 0.5073093094324904
Fold 3 precision: 0.4893076952712046
Fold 3 recall: 0.5850967126884183
--------------------------------
Fold 4 f1: 0.485577721222664
Fold 4 precision: 0.4681724608524479
Fold 4 recall: 0.5663985537684143
--------------------------------
Fold 5 f1: 0.495684414123275
Fold 5 precision: 0.4750196890451744
Fold 5 recall: 0.5808826719810863
--------------------------------
Fold 6 f1: 0.505226433146567
Fold 6 precision: 0.4897209688650607
Fold 6 recall: 0.5751957110004495
--------------------------------
